# Break → Fix

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Break → Fix

> **Problem.** A retrieval system ranks documents by dot product. Someone "boosts" a document by scaling its vector, or two embedding batches come back with different norms. Long or boosted vectors now win every query regardless of meaning, and the top result is confidently wrong.

**Idea.** Rank by cosine similarity (or normalise at insert and use dot product) so magnitude cannot game the ranking.

**Use when** you see irrelevant documents ranking first after any change to how vectors are produced or stored.  
**Not when** —.

```
query ─┐              dot        cosine
       ├─ doc0 (relevant)      0.62       0.62   ✓
       ├─ doc1 (refunds)       0.18       0.18
       └─ doc2 (×5 scaled)     2.41 ✗     0.48   ✓
```

**How it works.**
1. Three support documents and a query are embedded; document 2 is scaled ×5 to simulate a boost or a norm mismatch.
2. Ranking by dot product puts the scaled document first: its magnitude, not its meaning, produced the score.
3. Ranking by cosine divides magnitude out; the truly relevant document is first again.
4. Normalising every vector at insert time prevents the problem from ever arising.

| | what happens | result |
|:--|:--|:--|
| ✗ dot product | scaled doc | wrong document first |
| ✓ cosine | same vectors | relevant document first |

**Production code and its real output**

In [2]:
# BREAK: ranking documents by dot product when one vector was scaled — magnitude wins over meaning.
# FIX: cosine similarity normalises magnitude away.
from sklearn.metrics.pairwise import cosine_similarity

query_text = "how do I reset my password"
documents = [
    "To reset your password, open Settings and choose Forgot password.",
    "Our refund policy allows returns within 30 days of purchase.",
    "The password reset link expires after 15 minutes. Request a new one if needed.",
]
vectors = embed([query_text] + documents)
query, docs = vectors[0], vectors[1:]
docs_scaled = docs * np.array([1.0, 1.0, 5.0])[:, None]  # a "boost" someone applied to document 3

dot_scores = docs_scaled @ query
cosine_scores = cosine_similarity(docs_scaled, query[None, :])[:, 0]
print(f"{'':<5}{'dot':>8}{'cosine':>8}  document")
for index in range(3):
    print(
        
            f"doc{index:<2}{dot_scores[index]:>8.3f}{cosine_scores[index]:>8.3f}  "
            f"{documents[index][:55]}"
        
    )
print(
    "BREAK top by dot:   ",
    int(np.argmax(dot_scores)),
    "| FIX top by cosine:",
    int(np.argmax(cosine_scores)),
)
assert int(np.argmax(dot_scores)) == 2 and int(np.argmax(cosine_scores)) == 0

          dot  cosine  document
doc0    0.575   0.575  To reset your password, open Settings and choose Forgot
doc1    0.098   0.098  Our refund policy allows returns within 30 days of purc
doc2    2.510   0.502  The password reset link expires after 15 minutes. Reque
BREAK top by dot:    2 | FIX top by cosine: 0


**What the output shows.** By dot product the ×5 document ranked first; by cosine the password-reset document did — the same vectors, a different metric, a different answer.

**In practice**
- **normalise at insert** — and assert the norm on every write; a batch with a different norm is a silent ranking bug.
- **check after model changes** — new embedding model = new norms; re-embed the whole index.

**Alternatives** — euclidean distance on normalised vectors (same ranking)

**Terms** — *norm mismatch*: vectors of different lengths in one index
